# 8. Egresos hospitalarios y olas de calor — RM 60+

Análisis de hospitalizaciones en la Región Metropolitana durante años con olas de calor severas.
A diferencia de las defunciones (notebook 6), los egresos tienen resolución **anual** (no diaria),
por lo que el análisis es de comparación entre años, no por evento específico.

**Hipótesis:** Los años con olas de calor severas muestran mayor tasa de hospitalizaciones en
diagnósticos sensibles al calor: insuficiencia renal aguda (N17), deshidratación (E86),
trastornos electrolíticos (E87), e IAM (I21-I22).

**Fuentes:**
- Egresos DEIS: 2018 (baseline), 2019, 2022, 2023, 2024, 2025
- Años con olas conocidas: 2019 (+318 muertes), 2022 (+529), 2023 (+650), 2024 (+602)
- Nota: 2020 y 2021 no disponibles en DEIS (años COVID con estructura distinta)

**Limitación principal:** sin columna de fecha → solo comparación anual, no por evento.
El grupo 60-69 incluye 60-64 años (fuera del rango objetivo 65+) pero no hay forma de separarlo.

---

## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats as sp
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

DATA_DIR = '../data/egresos/'
OUT_DIR  = '../outputs/'

# Nombres cambiaron entre años — mapeo explícito
ARCHIVOS = {
    2018: f'{DATA_DIR}EGRE_DATOS_ABIERTOS_2018.csv',
    2019: f'{DATA_DIR}EGRE_DATOS_ABIERTOS_2019.csv',
    2022: f'{DATA_DIR}EGRE_DATOS_ABIERTOS_2022.csv',
    2023: f'{DATA_DIR}EGRESOS_2023.csv',
    2024: f'{DATA_DIR}EGR_DATOS_ABIERTO_2024.csv',
    2025: f'{DATA_DIR}EGR_DATOS_ABIERTO_2025.csv',
}
# Nota: 2020 y 2021 no disponibles en DEIS open data

GRUPOS_60PLUS = ['60 a 69', '70 a 79', '80 a 89', '90 y más']

# Columnas mínimas necesarias (presentes en todos los años)
USECOLS = ['REGION_RESIDENCIA', 'GRUPO_EDAD', 'ANO_EGRESO',
           'DIAG1', 'DIAG2', 'DIAS_ESTADA', 'CONDICION_EGRESO']

DIAG_CALOR = {
    'Insuf. renal aguda (N17)':    ('N17',),
    'Deshidratación (E86)':         ('E86',),
    'Trastorno electrolitos (E87)': ('E87',),
    'IAM (I21-I22)':                ('I21', 'I22'),
    'AVE (I60-I69)':                ('I60','I61','I62','I63','I64','I65','I66','I67','I68','I69'),
    'Insuf. cardíaca (I50)':        ('I50',),
    'Neumonía (J12-J18)':           ('J12','J13','J14','J15','J16','J17','J18'),
    'Sepsis (A40-A41)':             ('A40','A41'),
    'Calor directo (T67/X30)':      ('T67','X30','W92'),
    'Total 60+':                    None,
}

ANIO_BASE = 2018

OLAS_INFO = {
    2019: 'Ene 2019 (+318 muertes, 9d)',
    2022: 'Dic 2022 (+529 muertes, 8d)',
    2023: 'Feb 2023 (+650 muertes, 15d)',
    2024: 'Ene 2024 (+602 muertes, 23d)',
    2025: 'Feb 2025 (+356 muertes, 7d)',
}

COLORES_ANIO = {
    2019: '#e07b3c',
    2022: '#8e44ad',
    2023: '#c0392b',
    2024: '#2980b9',
    2025: '#27ae60',
}

print('Configuración lista — años:', list(ARCHIVOS.keys()))

## 1. Carga y filtro RM 60+

In [ ]:
import os

def cargar_egreso(anio, path):
    if not os.path.exists(path):
        print(f'  {anio}: NO ENCONTRADO — {path}')
        return None
    df = pd.read_csv(path, encoding='latin-1', sep=';', low_memory=False,
                     usecols=lambda c: c in USECOLS)
    mask_rm  = df['REGION_RESIDENCIA'].astype(str) == '13'
    mask_age = df['GRUPO_EDAD'].isin(GRUPOS_60PLUS)
    sub = df[mask_rm & mask_age].copy()
    sub['anio'] = anio
    print(f'  {anio}: {len(df):,} total | {len(sub):,} RM 60+ ({len(sub)/len(df)*100:.1f}%)')
    return sub

print('Cargando egresos...')
partes = []
for anio, path in ARCHIVOS.items():
    sub = cargar_egreso(anio, path)
    if sub is not None:
        partes.append(sub)

df_all = pd.concat(partes, ignore_index=True)
anios = sorted(df_all['anio'].unique())
print(f'\nTotal RM 60+: {len(df_all):,} egresos | años: {anios}')
print()
print('Egresos RM 60+ por año:')
print(df_all.groupby('anio').size().to_string())

## 2. Conteos anuales por diagnóstico

In [ ]:
# Contar egresos por año y diagnóstico
anios = sorted(df_all['anio'].unique())
resultados = []

for nombre, prefijos in DIAG_CALOR.items():
    for anio in anios:
        sub_a = df_all[df_all['anio'] == anio]
        if prefijos is None:
            n = len(sub_a)
        else:
            mask = sub_a['DIAG1'].str.startswith(prefijos, na=False)
            n = mask.sum()
        resultados.append({'diagnostico': nombre, 'anio': anio, 'n': n})

df_res = pd.DataFrame(resultados)
tabla = df_res.pivot(index='diagnostico', columns='anio', values='n')
# Calcular % cambio respecto a 2018
for a in anios:
    if a != ANIO_BASE:
        tabla[f'pct_{a}'] = ((tabla[a] - tabla[ANIO_BASE]) / tabla[ANIO_BASE] * 100).round(1)

print('Conteos RM 60+ y % cambio vs 2018:')
print(tabla.to_string())

# Guardar tabla
tabla.to_csv(f'{OUT_DIR}egr_conteos_anuales.csv')

## 3. Test estadístico: ¿el año caluroso tiene más ingresos?

In [ ]:
# Comparación 2019, 2022, 2023 vs 2018 para cada diagnóstico
# Usando test chi-cuadrado: n_diag / n_total en año caluroso vs año base

n_total = df_res[df_res['diagnostico'] == 'Total 60+'].set_index('anio')['n'].to_dict()
diags_test = [d for d in DIAG_CALOR if d != 'Total 60+']

test_rows = []
for diag in diags_test:
    n_base = df_res[(df_res['diagnostico'] == diag) & (df_res['anio'] == ANIO_BASE)]['n'].values[0]
    tot_base = n_total[ANIO_BASE]
    for anio_comp in [a for a in anios if a != ANIO_BASE]:
        n_comp = df_res[(df_res['diagnostico'] == diag) & (df_res['anio'] == anio_comp)]['n'].values[0]
        tot_comp = n_total[anio_comp]
        # Chi-cuadrado 2x2: diag vs no-diag, año base vs año caluroso
        tabla_2x2 = [[n_comp,           tot_comp - n_comp],
                     [n_base,           tot_base - n_base]]
        _, p, _, _ = sp.chi2_contingency(tabla_2x2)
        rr = (n_comp / tot_comp) / (n_base / tot_base)
        pct = (rr - 1) * 100
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
        test_rows.append({'diagnostico': diag, 'anio': anio_comp,
                          'n_base': n_base, 'n_comp': n_comp,
                          'RR': round(rr, 3), 'pct': round(pct, 1), 'p': p, 'sig': sig})

df_test = pd.DataFrame(test_rows)
print('RR y significancia (vs 2018 baseline):')
pivot_rr  = df_test.pivot(index='diagnostico', columns='anio', values='pct')
pivot_sig = df_test.pivot(index='diagnostico', columns='anio', values='sig')
for col in pivot_rr.columns:
    pivot_rr[f'{col}sig'] = pivot_sig[col]
print(pivot_rr.to_string(float_format='%.1f'))

## 4. Visualización: % cambio vs 2018 por diagnóstico y año

In [ ]:
diags_plot = [d for d in diags_test if d != 'Calor directo (T67/X30)']
anios_comp = [a for a in anios if a != ANIO_BASE]
n_comp = len(anios_comp)
width = 0.8 / n_comp

x = np.arange(len(diags_plot))

fig, ax = plt.subplots(figsize=(15, 6))
for i, anio in enumerate(anios_comp):
    pcts = [df_test[(df_test['diagnostico'] == d) & (df_test['anio'] == anio)]['pct'].values[0]
            for d in diags_plot]
    sigs = [df_test[(df_test['diagnostico'] == d) & (df_test['anio'] == anio)]['sig'].values[0]
            for d in diags_plot]
    offset = (i - (n_comp - 1) / 2) * width
    bars = ax.bar(x + offset, pcts, width * 0.92,
                  label=f'{anio} — {OLAS_INFO.get(anio, str(anio))}',
                  color=COLORES_ANIO.get(anio, 'grey'), alpha=0.85, edgecolor='white')
    for bar, sig in zip(bars, sigs):
        if sig:
            ypos = bar.get_height() + 0.4 if bar.get_height() >= 0 else bar.get_height() - 2.5
            ax.text(bar.get_x() + bar.get_width()/2, ypos, sig,
                    ha='center', fontsize=7, color='#333')

ax.axhline(0, color='black', linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels([d.split('(')[0].strip() for d in diags_plot],
                    rotation=30, ha='right', fontsize=9)
ax.set_ylabel('% cambio vs 2018 (baseline)')
ax.set_title('Egresos hospitalarios RM 60+ — % cambio vs 2018\n'
             'por diagnóstico y año (* p<0.05, ** p<0.01, *** p<0.001)')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}egr_cambio_anual.png', dpi=150, bbox_inches='tight')
plt.show()

# Calor directo por separado (n muy pequeño)
calor_rows = df_test[df_test['diagnostico'] == 'Calor directo (T67/X30)']
print('\nCalor directo T67/X30 (n absolutos):')
for _, row in calor_rows.iterrows():
    print(f'  {int(row["anio"])}: n_base={row["n_base"]}  n_comp={row["n_comp"]}  RR={row["RR"]:.2f}  p={row["p"]:.3f}')

## 5. Mortalidad intrahospitalaria por diagnóstico y año

In [ ]:
# CONDICION_EGRESO == 2 → fallecido
mort_rows = []
for nombre, prefijos in DIAG_CALOR.items():
    if nombre == 'Total 60+':
        continue
    for anio in anios:
        sub_a = df_all[df_all['anio'] == anio]
        sub_d = sub_a if prefijos is None else sub_a[sub_a['DIAG1'].str.startswith(prefijos, na=False)]
        n_ing  = len(sub_d)
        n_mort = (sub_d['CONDICION_EGRESO'] == 2).sum()
        tasa   = n_mort / n_ing * 100 if n_ing > 0 else np.nan
        mort_rows.append({'diagnostico': nombre, 'anio': anio,
                          'n_ingresos': n_ing, 'n_muertes': n_mort, 'tasa_mort': tasa})

df_mort = pd.DataFrame(mort_rows)
pivot_mort = df_mort.pivot(index='diagnostico', columns='anio', values='tasa_mort')

diags_mort = [d for d in diags_test if d != 'Calor directo (T67/X30)']
x = np.arange(len(diags_mort))
n_anios = len(anios)
w = 0.8 / n_anios

fig, ax = plt.subplots(figsize=(14, 6))
for i, anio in enumerate(anios):
    tasas = [pivot_mort.loc[d, anio] if d in pivot_mort.index else np.nan for d in diags_mort]
    color = '#bdc3c7' if anio == ANIO_BASE else COLORES_ANIO.get(anio, 'grey')
    lbl   = f'{anio} (baseline)' if anio == ANIO_BASE else f'{anio}'
    ax.bar(x + (i - (n_anios-1)/2) * w, tasas, w*0.92,
           label=lbl, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([d.split('(')[0].strip() for d in diags_mort], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Mortalidad intrahospitalaria (%)')
ax.set_title('Tasa de mortalidad intrahospitalaria RM 60+\npor diagnóstico y año')
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}egr_mortalidad_intra.png', dpi=150, bbox_inches='tight')
plt.show()

print('Tasas mortalidad intrahospitalaria (%):')
print(pivot_mort.round(2).to_string())

## 6. Días de estadía: ¿más graves en años calurosos?

In [ ]:
diags_estada = [
    ('Insuf. renal aguda (N17)', ('N17',)),
    ('IAM (I21-I22)',            ('I21','I22')),
    ('AVE (I60-I69)',            ('I60','I61','I62','I63','I64','I65','I66','I67','I68','I69')),
    ('Neumonía (J12-J18)',       ('J12','J13','J14','J15','J16','J17','J18')),
]

fig, axes = plt.subplots(1, len(diags_estada), figsize=(16, 5))
colores_box = ['#bdc3c7'] + [COLORES_ANIO.get(a, 'steelblue') for a in anios if a != ANIO_BASE]

for ax, (nombre, prefijos) in zip(axes, diags_estada):
    grupos, labels, medias = [], [], []
    for anio in anios:
        sub_d = df_all[(df_all['anio'] == anio) &
                       df_all['DIAG1'].str.startswith(prefijos, na=False)]
        dias = sub_d['DIAS_ESTADA'].dropna().clip(0, 30).values
        grupos.append(dias)
        labels.append(str(anio))
        medias.append(dias.mean() if len(dias) > 0 else 0)

    stat, p_kw = sp.kruskal(*[g for g in grupos if len(g) > 0])
    sig_kw = '***' if p_kw < 0.001 else ('**' if p_kw < 0.01 else ('*' if p_kw < 0.05 else 'n.s.'))

    bp = ax.boxplot(grupos, labels=labels, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(linewidth=0.8),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))
    for patch, color in zip(bp['boxes'], colores_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    ax.set_title(nombre.split('(')[0].strip(), fontsize=10, fontweight='bold')
    ax.set_ylabel('Días de estadía' if ax == axes[0] else '')
    ax.set_xlabel(f'KW: {sig_kw} (p={p_kw:.4f})', fontsize=8)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    for j, m in enumerate(medias):
        ax.text(j+1, ax.get_ylim()[1]*0.92, f'x̄={m:.1f}', ha='center', fontsize=7, color='#333')

fig.suptitle('Días de estadía RM 60+ por diagnóstico y año (clip 0-30d)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}egr_dias_estadia.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Diagnósticos directos de calor: T67, X30, E86 — serie anual

In [ ]:
grupos_calor = {
    'T67 (golpe calor)':    ('T67',),
    'X30 (calor natural)':  ('X30',),
    'E86 (deshidratación)': ('E86',),
    'E87 (electrolitos)':   ('E87',),
    'N17 (renal aguda)':    ('N17',),
}

fig, axes = plt.subplots(1, len(grupos_calor), figsize=(17, 4))

for ax, (nombre, prefijos) in zip(axes, grupos_calor.items()):
    counts = []
    for anio in anios:
        n = df_all[(df_all['anio'] == anio) &
                   df_all['DIAG1'].str.startswith(prefijos, na=False)].shape[0]
        counts.append(n)

    colores_b = ['#bdc3c7' if a == ANIO_BASE else COLORES_ANIO.get(a, 'steelblue') for a in anios]
    bars = ax.bar([str(a) for a in anios], counts, color=colores_b, edgecolor='white', alpha=0.88)
    for bar, val in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                str(val), ha='center', fontsize=9, fontweight='bold')
    ax.set_title(nombre, fontsize=10, fontweight='bold')
    ax.set_ylabel('N egresos RM 60+' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=40, labelsize=8)
    ax.set_ylim(0, max(counts) * 1.18)

fig.suptitle('Egresos RM 60+ con diagnósticos sensibles al calor — serie 2018–2025\n'
             '(gris=baseline 2018, colores=años con olas conocidas)',
             fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}egr_diag_calor.png', dpi=150, bbox_inches='tight')
plt.show()

print('Serie anual:')
for nombre, prefijos in grupos_calor.items():
    counts = {a: df_all[(df_all['anio']==a) & df_all['DIAG1'].str.startswith(prefijos, na=False)].shape[0] for a in anios}
    print(f'  {nombre}: {counts}')